# Object Detection Project

## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/HoangDuonng1359/object-detection"
PROJECT_DIR = Path("/kaggle/working/object-detection")

IMAGE_SIZE = 512
EPOCHS = 70
BATCH_SIZE = 4
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_PRETRAINED_BACKBONE = True
FREEZE_BACKBONE_STEM = False
OVERSAMPLE_CLASSES = ["chair"]
OVERSAMPLE_FACTOR = 2.0

# Set these to small values for a quick smoke test. Use 0 for full train/validation.
MAX_TRAIN_BATCHES = 0
MAX_VAL_BATCHES = 0

CONF_THRESHOLD = 0.25
# Tuned from validation: chair has low recall, car has room to trade precision for recall.
CLASS_THRESHOLDS = {
    "person": 0.25,
    "car": 0.20,
    "dog": 0.25,
    "cat": 0.25,
    "chair": 0.10,
}
CLASS_THRESHOLDS_TEXT = ",".join(f"{name}={value}" for name, value in CLASS_THRESHOLDS.items())
NMS_THRESHOLD = 0.35
MAX_DETECTIONS = 100

RUN_THRESHOLD_SWEEP = True
CLASS_THRESHOLD_CONFIGS = [
    {
        "name": "balanced",
        "default": 0.25,
        "thresholds": CLASS_THRESHOLDS,
    },
    {
        "name": "more_recall",
        "default": 0.20,
        "thresholds": {"person": 0.20, "car": 0.15, "dog": 0.20, "cat": 0.20, "chair": 0.08},
    },
    {
        "name": "clean",
        "default": 0.30,
        "thresholds": {"person": 0.30, "car": 0.25, "dog": 0.25, "cat": 0.30, "chair": 0.15},
    },
    {
        "name": "map_oriented",
        "default": 0.15,
        "thresholds": {"person": 0.15, "car": 0.15, "dog": 0.15, "cat": 0.15, "chair": 0.08},
    },
]
NMS_VALUES = [0.35, 0.45, 0.50]

# Optional hidden/test image directory. Leave as None if you only want val predictions.
TEST_IMAGE_DIR = None
FINAL_PREDICTIONS = Path("/kaggle/working/predictions.json")


## 2. Clone Code And Install Requirements

In [2]:
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
        raise ValueError("Edit REPO_URL in the settings cell before cloning.")
    clone_cmd = ["git", "clone"]
    clone_cmd += [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(clone_cmd, check=True)
else:
    print(f"Project directory already exists: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

requirements = PROJECT_DIR / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

import torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())


Cloning into '/kaggle/working/object-detection'...


cwd: /kaggle/working/object-detection
python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda_available: True


## 3. Locate Kaggle Input Dataset

In [3]:
PUBLIC_DIR = Path("/kaggle/input/datasets/duong1589/object-detection/public")
TRAIN_JSON = PUBLIC_DIR / "annotations" / "train.json"
VAL_JSON = PUBLIC_DIR / "annotations" / "val.json"
TRAIN_IMAGES = PUBLIC_DIR / "train" / "images"
VAL_IMAGES = PUBLIC_DIR / "val" / "images"
EVALUATOR = PUBLIC_DIR / "tools" / "evaluate_predictions.py"

print("PUBLIC_DIR:", PUBLIC_DIR)
print("TRAIN_JSON:", TRAIN_JSON)
print("VAL_JSON:", VAL_JSON)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_IMAGES:", VAL_IMAGES)
print("EVALUATOR:", EVALUATOR, EVALUATOR.exists())


PUBLIC_DIR: /kaggle/input/datasets/duong1589/object-detection/public
TRAIN_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json
VAL_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json
TRAIN_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/train/images
VAL_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/val/images
EVALUATOR: /kaggle/input/datasets/duong1589/object-detection/public/tools/evaluate_predictions.py True


## 4. Sanity Check Dataset And Code

In [4]:
import json

for split_name, json_path in [("train", TRAIN_JSON), ("val", VAL_JSON)]:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(split_name, "images=", len(data["images"]), "annotations=", len(data["annotations"]))

subprocess.run([sys.executable, "-m", "py_compile", "train.py", "predict.py"], check=True)
subprocess.run([
    sys.executable, "-m", "utils.check_dataset",
    "--annotations", str(TRAIN_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--image_size", str(IMAGE_SIZE),
    "--batch_size", "2",
    "--train",
], check=True)


train images= 7500 annotations= 10642
val images= 1500 annotations= 2021
dataset_size=7500
classes=['person', 'car', 'dog', 'cat', 'chair']
sample_image_shape=(3, 512, 512)
sample_image_id=img_000c52c6d12f.jpg
sample_boxes=(1, 4)
batch_image_shape=(2, 3, 512, 512)
batch_box_counts=[1, 1]


CompletedProcess(args=['/usr/bin/python3', '-m', 'utils.check_dataset', '--annotations', '/kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json', '--image_dir', '/kaggle/input/datasets/duong1589/object-detection/public/train/images', '--image_size', '512', '--batch_size', '2', '--train'], returncode=0)

## 5. Train

In [5]:
from IPython.display import display
import pandas as pd

train_cmd = [
    sys.executable, "train.py",
    "--train_data", str(TRAIN_JSON),
    "--val_data", str(VAL_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--val_image_dir", str(VAL_IMAGES),
    "--checkpoint_dir", str(PROJECT_DIR / "models"),
    "--image_size", str(IMAGE_SIZE),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--class_thresholds", CLASS_THRESHOLDS_TEXT,
    "--nms_threshold", str(NMS_THRESHOLD),
]
if USE_AMP:
    train_cmd.append("--amp")
if USE_PRETRAINED_BACKBONE:
    train_cmd.append("--pretrained_backbone")
if FREEZE_BACKBONE_STEM:
    train_cmd.append("--freeze_backbone_stem")
if OVERSAMPLE_CLASSES and OVERSAMPLE_FACTOR > 1.0:
    train_cmd += ["--oversample_classes", *OVERSAMPLE_CLASSES]
    train_cmd += ["--oversample_factor", str(OVERSAMPLE_FACTOR)]
if MAX_TRAIN_BATCHES > 0:
    train_cmd += ["--max_train_batches", str(MAX_TRAIN_BATCHES)]
if MAX_VAL_BATCHES > 0:
    train_cmd += ["--max_val_batches", str(MAX_VAL_BATCHES)]

print("Train command:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

history_files = sorted((PROJECT_DIR / "models").glob("train_history_*.csv"), key=lambda item: item.stat().st_mtime)
if history_files:
    HISTORY_CSV = history_files[-1]
    history_df = pd.read_csv(HISTORY_CSV)
    print("History CSV:", HISTORY_CSV)
    display_columns = [
        "epoch", "lr", "train_loss", "val_loss", "val_map50", "best_map50",
        "train_box_loss", "train_objectness_loss", "train_class_loss",
        "val_box_loss", "val_objectness_loss", "val_class_loss",
    ]
    display(history_df[[col for col in display_columns if col in history_df.columns]].tail(10))
else:
    HISTORY_CSV = None
    print("No train history CSV found.")


Train command:
/usr/bin/python3 train.py --train_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json --val_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json --image_dir /kaggle/input/datasets/duong1589/object-detection/public/train/images --val_image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --checkpoint_dir /kaggle/working/object-detection/models --image_size 512 --epochs 70 --batch_size 8 --num_workers 2 --lr 0.0001 --weight_decay 0.0001 --conf_threshold 0.25 --nms_threshold 0.35 --amp --pretrained_backbone --oversample_classes chair --oversample_factor 2.0
device=cuda amp=True
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 172MB/s]


epoch=1/70 lr=6.66667e-05 train_loss=2.9484 val_loss=2.5634 val_map50=0.4326 best_map50=0.4326


epoch=2/70 lr=0.0001 train_loss=2.4307 val_loss=2.2591 val_map50=0.4849 best_map50=0.4849


epoch=3/70 lr=0.0001 train_loss=2.2806 val_loss=2.2234 val_map50=0.4934 best_map50=0.4934


...

epoch=67/70 lr=4.93874e-07 train_loss=0.9950 val_loss=1.8491 val_map50=0.6617 best_map50=0.6629


epoch=68/70 lr=2.19701e-07 train_loss=0.9974 val_loss=1.8479 val_map50=0.6577 best_map50=0.6629


epoch=69/70 lr=5.49554e-08 train_loss=0.9945 val_loss=1.8527 val_map50=0.6585 best_map50=0.6629


epoch=70/70 lr=0 train_loss=1.0000 val_loss=1.8491 val_map50=0.6557 best_map50=0.6629
best_checkpoint=/kaggle/working/object-detection/models/best.pth
history=/kaggle/working/object-detection/models/train_history_20260530_153319.csv
History CSV: /kaggle/working/object-detection/models/train_history_20260530_153319.csv


,epoch,lr,train_loss,val_loss,val_map50,best_map50,train_box_loss,train_objectness_loss,train_class_loss,val_box_loss,val_objectness_loss,val_class_loss
60,61,4.386522e-06,1.011797,1.847393,0.656154,0.662937,0.195681,0.030067,0.006654,0.284082,0.286727,0.280516
61,62,3.476735e-06,1.003026,1.854049,0.654049,0.662937,0.193946,0.030198,0.006198,0.284163,0.288199,0.290068
62,63,2.669216e-06,1.006582,1.849231,0.653080,0.662937,0.194759,0.029594,0.006390,0.283905,0.286278,0.286855
63,64,1.965741e-06,0.997952,1.845999,0.659717,0.662937,0.192851,0.028988,0.009421,0.284310,0.286206,0.276486
64,65,1.367855e-06,0.999942,1.856804,0.655097,0.662937,0.193859,0.027089,0.007114,0.284481,0.290451,0.287899
65,66,8.768729e-07,0.995387,1.852243,0.656302,0.662937,0.192583,0.029084,0.006777,0.284404,0.288235,0.283972
66,67,4.938743e-07,0.994972,1.849066,0.661721,0.662937,0.192754,0.027859,0.006687,0.283671,0.285191,0.291044
67,68,2.197009e-07,0.997406,1.847941,0.657673,0.662937,0.193031,0.029398,0.005707,0.284083,0.285416,0.284216
68,69,5.495543e-08,0.994502,1.852651,0.658469,0.662937,0.192551,0.028536,0.006419,0.284272,0.287979,0.286626
69,70,0.000000e+00,1.000018,1.849138,0.655664,0.662937,0.193711,0.028704,0.005517,0.283837,0.285322,0.289258


## 6. Predict On Validation And Evaluate

In [6]:
VAL_PREDICTIONS = PROJECT_DIR / "val_predictions.json"
predict_val_cmd = [
    sys.executable, "predict.py",
    "--image_dir", str(VAL_IMAGES),
    "--output", str(VAL_PREDICTIONS),
    "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
    "--batch_size", str(BATCH_SIZE),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--class_thresholds", CLASS_THRESHOLDS_TEXT,
    "--nms_threshold", str(NMS_THRESHOLD),
    "--max_detections", str(MAX_DETECTIONS),
]
print("Predict validation command:")
print(" ".join(predict_val_cmd))
subprocess.run(predict_val_cmd, check=True)

val_predictions_data = json.loads(VAL_PREDICTIONS.read_text(encoding="utf-8"))
num_pred_boxes = sum(len(item["boxes"]) for item in val_predictions_data)
print(f"Validation predictions: images={len(val_predictions_data)} boxes={num_pred_boxes}")
print("Prediction preview:")
print(json.dumps(val_predictions_data[:3], ensure_ascii=False, indent=2))

if EVALUATOR.exists():
    VAL_SCORE = PROJECT_DIR / "val_score.json"
    subprocess.run([
        sys.executable, str(EVALUATOR),
        "--ground_truth", str(VAL_JSON),
        "--predictions", str(VAL_PREDICTIONS),
        "--output", str(VAL_SCORE),
    ], check=True)
    score = json.loads(VAL_SCORE.read_text(encoding="utf-8"))
    print("Validation score:")
    print(json.dumps(score, ensure_ascii=False, indent=2))
    if "per_class" in score:
        per_class_df = pd.DataFrame.from_dict(score["per_class"], orient="index")
        display(per_class_df)
else:
    VAL_SCORE = None
    print("Evaluator not found, skipped local validation scoring.")


Predict validation command:
/usr/bin/python3 predict.py --image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --output /kaggle/working/object-detection/val_predictions.json --checkpoint /kaggle/working/object-detection/models/best.pth --batch_size 8 --conf_threshold 0.25 --nms_threshold 0.35 --max_detections 100
wrote 1500 predictions to /kaggle/working/object-detection/val_predictions.json
Validation predictions: images=1500 boxes=1955
Prediction preview:
[
  {
    "image_id": "img_00090df9158f.jpg",
    "boxes": [
      {
        "class": "dog",
        "confidence": 0.459243,
        "bbox": [
          127.03,
          53.59,
          495.64,
          375.0
        ]
      }
    ]
  },
  {
    "image_id": "img_003c3ddc6a82.jpg",
    "boxes": [
      {
        "class": "dog",
        "confidence": 0.313938,
        "bbox": [
          6.76,
          69.98,
          496.82,
          367.19
        ]
      }
    ]
  },
  {
    "image_id": "img_01073356e

,ap,num_ground_truth,num_predictions,true_positives,false_positives,recall,precision
person,0.738190,1074,1061,822,239,0.765363,0.774741
car,0.606976,283,260,185,75,0.653710,0.711538
dog,0.743624,206,212,161,51,0.781553,0.759434
cat,0.758297,176,158,138,20,0.784091,0.873418
chair,0.467597,282,264,151,113,0.535461,0.571970


## 7. Threshold Sweep On Validation


In [7]:
import json
import pandas as pd

BEST_CHECKPOINT = PROJECT_DIR / "models" / "best.pth"

def thresholds_to_text(thresholds):
    return ",".join(f"{name}={value}" for name, value in thresholds.items())

if RUN_THRESHOLD_SWEEP:
    if not (VAL_JSON.exists() and EVALUATOR.exists()):
        raise FileNotFoundError("Need VAL_JSON and EVALUATOR for threshold sweep.")
    rows = []
    sweep_dir = PROJECT_DIR / "threshold_sweep"
    sweep_dir.mkdir(parents=True, exist_ok=True)
    for config in CLASS_THRESHOLD_CONFIGS:
        config_name = config["name"]
        default_conf = config["default"]
        thresholds = config["thresholds"]
        thresholds_text = thresholds_to_text(thresholds)
        safe_thresholds = thresholds_text.replace(",", "_").replace("=", "-")
        for nms in NMS_VALUES:
            sweep_pred = sweep_dir / f"predictions_{config_name}_nms{nms:.2f}_{safe_thresholds}.json"
            sweep_score = sweep_dir / f"score_{config_name}_nms{nms:.2f}_{safe_thresholds}.json"
            print(f"Running config={config_name} default={default_conf:.2f} thresholds={thresholds_text} nms={nms:.2f}")
            !{sys.executable} predict.py --image_dir "{VAL_IMAGES}" --output "{sweep_pred}" --checkpoint "{BEST_CHECKPOINT}" --batch_size {BATCH_SIZE} --conf_threshold {default_conf} --class_thresholds "{thresholds_text}" --nms_threshold {nms} --max_detections {MAX_DETECTIONS}
            !{sys.executable} "{EVALUATOR}" --ground_truth "{VAL_JSON}" --predictions "{sweep_pred}" --output "{sweep_score}"
            score = json.loads(sweep_score.read_text(encoding="utf-8"))
            per_class = score.get("per_class", {})
            chair = per_class.get("chair", {})
            rows.append({
                "config": config_name,
                "thresholds": thresholds_text,
                "default_conf": default_conf,
                "nms": nms,
                "map50": score.get("mAP@0.5", 0.0),
                "performance_points": score.get("performance_points", 0),
                "precision": score.get("micro_precision", 0.0),
                "recall": score.get("micro_recall", 0.0),
                "num_predictions": score.get("num_predictions", 0),
                "chair_ap": chair.get("ap", 0.0),
                "chair_precision": chair.get("precision", 0.0),
                "chair_recall": chair.get("recall", 0.0),
            })
    sweep_df = pd.DataFrame(rows).sort_values(["map50", "precision"], ascending=[False, False])
    sweep_csv = sweep_dir / "threshold_sweep_results.csv"
    sweep_df.to_csv(sweep_csv, index=False)
    print("Threshold sweep results:", sweep_csv)
    display(sweep_df)
else:
    print("Threshold sweep disabled. Set RUN_THRESHOLD_SWEEP = True to run it.")


Running conf=0.05 nms=0.35
wrote 1500 predictions to /kaggle/working/object-detection/threshold_sweep/predictions_conf0.05_nms0.35.json
{
  "mAP@0.5": 0.711577,
  "performance_points": 20,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 6181,
  "micro_precision": 0.270183,
  "micro_recall": 0.826324,
  "per_class": {
    "person": {
      "ap": 0.792964,
      "num_ground_truth": 1074,
      "num_predictions": 2977,
      "true_positives": 924,
      "false_positives": 2053,
      "recall": 0.860335,
      "precision": 0.31038
    },
    "car": {
      "ap": 0.661698,
      "num_ground_truth": 283,
      "num_predictions": 1079,
      "true_positives": 219,
      "false_positives": 860,
      "recall": 0.773852,
      "precision": 0.202966
    },
    "dog": {
      "ap": 0.775532,
      "num_ground_truth": 206,
      "num_predictions": 418,
      "true_positives": 171,
      "false_positives": 247,
      "recall": 0.830097,
      "precision": 0.409091
   

,conf,nms,map50,performance_points,precision,recall,num_predictions
1,0.05,0.45,0.716507,20,0.209557,0.850569,8203
0,0.05,0.35,0.711577,20,0.270183,0.826324,6181
2,0.05,0.50,0.706989,20,0.176155,0.860465,9872
5,0.10,0.45,0.702682,20,0.413819,0.803068,3922
4,0.10,0.35,0.697552,20,0.488143,0.784265,3247
6,0.10,0.50,0.695348,20,0.354414,0.812469,4633
9,0.15,0.45,0.690653,20,0.542215,0.775359,2890
8,0.15,0.35,0.687181,20,0.606551,0.760515,2534
10,0.15,0.50,0.684159,20,0.477526,0.783276,3315
13,0.20,0.45,0.678070,20,0.638423,0.753093,2384


## 8. Predict On Hidden/Test Images If Available


In [8]:
if TEST_IMAGE_DIR is None:
    print("TEST_IMAGE_DIR is None. Skipped final prediction.")
else:
    test_dir = Path(TEST_IMAGE_DIR)
    final_cmd = [
        sys.executable, "predict.py",
        "--image_dir", str(test_dir),
        "--output", str(FINAL_PREDICTIONS),
        "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
        "--batch_size", str(BATCH_SIZE),
        "--conf_threshold", str(CONF_THRESHOLD),
        "--class_thresholds", CLASS_THRESHOLDS_TEXT,
        "--nms_threshold", str(NMS_THRESHOLD),
    ]
    print("Final predict command:")
    print(" ".join(final_cmd))
    subprocess.run(final_cmd, check=True)
    final_data = json.loads(FINAL_PREDICTIONS.read_text(encoding="utf-8"))
    print(f"Final predictions: images={len(final_data)} boxes={sum(len(item['boxes']) for item in final_data)}")
    print("Final prediction preview:")
    print(json.dumps(final_data[:3], ensure_ascii=False, indent=2))
    print("Final predictions path:", FINAL_PREDICTIONS)


TEST_IMAGE_DIR is None. Skipped final prediction.


## 9. Collect Artifacts


In [9]:
import shutil

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    PROJECT_DIR / "models" / "best.pth",
    PROJECT_DIR / "models" / "last.pth",
    VAL_PREDICTIONS,
    PROJECT_DIR / "val_score.json",
]
files_to_copy += sorted((PROJECT_DIR / "models").glob("train_history_*.csv"))
sweep_results = PROJECT_DIR / "threshold_sweep" / "threshold_sweep_results.csv"
if sweep_results.exists():
    files_to_copy.append(sweep_results)
if FINAL_PREDICTIONS.exists():
    files_to_copy.append(FINAL_PREDICTIONS)

copied = []
for src in files_to_copy:
    if src.exists():
        dst = ARTIFACT_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst)

zip_path = shutil.make_archive("/kaggle/working/object_detection_artifacts", "zip", ARTIFACT_DIR)
print("Artifacts directory:", ARTIFACT_DIR)
print("Artifacts zip:", zip_path)
print("Files:")
artifact_rows = []
for path in sorted(ARTIFACT_DIR.iterdir()):
    artifact_rows.append({"file": path.name, "size_mb": round(path.stat().st_size / (1024 * 1024), 3)})
display(pd.DataFrame(artifact_rows))


Artifacts directory: /kaggle/working/artifacts
Artifacts zip: /kaggle/working/object_detection_artifacts.zip
Files:


,file,size_mb
0,best.pth,368.239
1,last.pth,368.239
2,threshold_sweep_results.csv,0.001
3,train_history_20260530_153319.csv,0.043
4,val_predictions.json,0.417
5,val_score.json,0.001


## 10. Summary Shown In Notebook


In [10]:
summary = {
    "best_checkpoint": str(PROJECT_DIR / "models" / "best.pth"),
    "history_csv": str(HISTORY_CSV) if HISTORY_CSV is not None else None,
    "val_predictions": str(VAL_PREDICTIONS),
    "val_score": str(PROJECT_DIR / "val_score.json"),
    "threshold_sweep_results": str(PROJECT_DIR / "threshold_sweep" / "threshold_sweep_results.csv"),
    "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

if HISTORY_CSV is not None:
    history_df = pd.read_csv(HISTORY_CSV)
    print("Final history row:")
    display(history_df.tail(1))

score_path = PROJECT_DIR / "val_score.json"
if score_path.exists():
    score = json.loads(score_path.read_text(encoding="utf-8"))
    print(f"mAP@0.5={score.get('mAP@0.5')} performance_points={score.get('performance_points')}")


{
  "best_checkpoint": "/kaggle/working/object-detection/models/best.pth",
  "history_csv": "/kaggle/working/object-detection/models/train_history_20260530_153319.csv",
  "val_predictions": "/kaggle/working/object-detection/val_predictions.json",
  "val_score": "/kaggle/working/object-detection/val_score.json",
  "threshold_sweep_results": "/kaggle/working/object-detection/threshold_sweep/threshold_sweep_results.csv",
  "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip"
}
Final history row:


,run_started_at,epoch,total_epochs,train_data,val_data,image_dir,val_image_dir,image_size,batch_size,num_workers,...,train_objectness_loss,train_class_loss,train_num_positive,val_loss,val_box_loss,val_objectness_loss,val_class_loss,val_num_positive,val_map50,best_map50
69,2026-05-30 15:33:19,70,70,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,512,8,2,...,0.028704,0.005517,104.48081,1.849138,0.283837,0.285322,0.289258,89.308511,0.655664,0.662937


mAP@0.5=0.662937 performance_points=20
